# Everytown dbt demo & discussion

This is a pointed discussion, demo, and Q&A session about using dbt in Civis Platform.

## Topics of Discussion

1. Setting up a dbt projects and profile configuration, including any templates or best practices for faster set ups
2. A suggested workflow for developing and productionalizing dbt, including when to use Civis Studio versus the dbt runner in a job or workflow
3. How retries work in Civis, including what happens to ephemeral models when a job fails and how to handle your hourly table build
4. ???

## 1. Project Setup

### a) Review profiles.yml. 
- Required for development in Civis Studio. Can be autogenerated when using a dbt platform job
- Use [our public profiles.yml](https://github.com/civisanalytics/platform-code-examples/blob/efe1a71e81da72d6093e18bced2effdb76cc0e8e/dbt/profiles.yml) to get started.

### b) Set up Civis Studio Parameters accordingly
- Dictated by profiles.yml
- Show parameters based on Civis's publicly available profiles.yml

In [1]:
!cat /workspace/platform-code-examples/dbt/profiles.yml

dbt-civis:
  target: redshift_adapter
  outputs:
    redshift_adapter:
      type: "{{ env_var('DATABASE_TYPE') }}"
      host: "{{ env_var('DATABASE_HOST') }}"
      port: "{{ env_var('DATABASE_PORT') | as_number }}"
      user: "{{ env_var('DATABASE_USERNAME') }}"
      pass: "{{ env_var('DBT_ENV_SECRET_DATABASE_PASSWORD') }}"
      dbname: "{{ env_var('DATABASE_NAME') }}"
      schema: "{{ env_var('DBT_SCHEMA') }}"
      threads: 4

    bigquery_service_account_adapter:
      type: "{{ env_var('DATABASE_TYPE') }}"
      method: "{{ env_var('GCP_AUTH_METHOD') }}"
      project: "{{ env_var('GCP_PROJECT_ID') }}"
      schema: "{{ env_var('DBT_SCHEMA') }}"
      keyfile: "{{ env_var('GCP_SERVICE_ACCOUNT_KEYFILE_JSON') }}"
      threads: 4

    bigquery_oauth_adapter:
      threads: 4 # Must be a value of 1 or greater
      type: "{{ env_var('DATABASE_TYPE') }}"
      method: "{{ env_var('GCP_AUTH_METHOD') }}"
      project: "{{ env_var('GCP_PROJECT_ID') }}"
      schema: "{{ env_var('D

### c) Execute dbt debug

1. CLI - Use a bash terminal to run commands. This offers the most flexibility.
2. dbt extension - Mostly nice for additional context on a dbt project. It can execute build/run/test commands, but not debug/retry

In [3]:
%%sh
cd /workspace/platform-code-examples/dbt
dbt debug

       dbt 2.0.9
   Loading profiles.yml
 Debugging profile: redshift_adapter
 Debugging dbt version: 2.0.9
 Debugging platform: linux x86_64 (unix)
 Debugging adapter type: redshift (remote)
 Debugging dependencies:
  git: OK
 Debugging connection:
  "port": 5432,
  "database": "dev",
  "schema": "jaffle_demo",
  "host": "redshift.acme.civis.io",
  "user": "jsmith"
 Debugging connection test: OK
  Debugged All checks passed!

==================== Execution Summary =====================
Finished 'debug' successfully for target 'redshift_adapter' s]


## 2. Building a project for productionalization

### Tips

- Use one big dbt project instead of multiple smaller ones to take advantage of code reuse
- Create a folder structure that makes sense for your domain. A good default option is stg/intermediate/marts to cover E/T/L
- Use tags to define and execute pipelines. 
- Use inheritence in `dbt_project.yml` to cut down on redundant configuration (show /workspace/platform-code-examples/dbt/dbt_project.yml)
- Use vars and macros to abstract functionality as much as is reasonable.